In [12]:
import stim
import pymatching
import numpy as np
from itertools import combinations

def run_lilliput(QEC_cycle, max_weight):
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        before_measure_flip_probability=0.02,
        distance=3,
        rounds=QEC_cycle,
       # before_round_data_depolarization=p
    )
    
    
    dem = circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(dem)
    sampler = circuit.compile_detector_sampler()
    num_det = dem.num_detectors
    MAX_SYNDROME_WEIGHT = max_weight  # feasible for d=3
    
    lut = {}
    
    for w in range(MAX_SYNDROME_WEIGHT + 1):
        for locs in combinations(range(num_det), w):
            syndrome = np.zeros(num_det, dtype=np.uint8)
            for i in locs:
                syndrome[i] = 1
    
            correction = matching.decode(syndrome, algorithm="mwpm")
            lut[tuple(syndrome.tolist())] = correction
    
    
    shots = 1000
    
    detectors, observables = sampler.sample(
        shots=shots,
        separate_observables=True,
    )
    
    zero_correction = np.zeros_like(next(iter(lut.values())))
    
    logical_errors = 0
    missing_count = 0
    
    for d, o in zip(detectors, observables):
        key = tuple(d.tolist())
    
        if key in lut:
            pred = lut[key]
        else:
            pred = zero_correction
            missing_count += 1
    
        if pred[0] != o[0]:
            logical_errors += 1
    return {
        "detectors": QEC_cycle, # *8
        "lut_entries": len(lut),
        "logical_error_rate": logical_errors / shots * 100,
        "missing_percent": missing_count / shots*100
    }


In [13]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Flowable
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import pagesizes
from reportlab.lib.units import inch
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
from reportlab.lib.styles import getSampleStyleSheet
import os

class DiagonalHeader(Flowable):
    def __init__(self, width=120, height=60):
        Flowable.__init__(self)
        self.width = width
        self.height = height

    def draw(self):
        c = self.canv
        c.setStrokeColor(colors.black)
        c.line(0, self.height, self.width, 0)
        c.setFont("Helvetica", 10)
        c.drawString(5, 5, "QEC Round")
        c.drawRightString(self.width - 5, self.height - 15, "Max Weight")

def generate_pdf():
    filename = "Lilliput_QEC_Results_3_002.pdf"
    doc = SimpleDocTemplate(filename, pagesize=pagesizes.A4)
    elements = []

    styles = getSampleStyleSheet()
    title_style = styles["Heading1"]

    elements.append(Paragraph("Lilliput (Distance 3 and phenomenological error prob = 0.02) Simulation Results", title_style))
    elements.append(Spacer(1, 0.3 * inch))

    qec_rounds = [2, 3, 5]
    max_weights = [1, 2, 3]

    # Table header
    diag_header = DiagonalHeader(width=120, height=60)
    table_data = [[diag_header, "1", "2", "3"]]

    for qec in qec_rounds:
        row = [str(qec)]
        for mw in max_weights:
            result = run_lilliput(qec, mw)
            det = result['detectors']*8
            cell_text = (
                f"Detectors: {det} ({result['detectors']} X 8)\n"
                f"LUT Length: {result['lut_entries']}\n"
                f"Logical Error: {float(result['logical_error_rate']):.4f}%\n"
                f"Miss: {float(result['missing_percent']):.4f}%"
            )

            row.append(cell_text)

        table_data.append(row)

    # Create table
    table = Table(table_data, colWidths=[1.8*inch]*4)

    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN', (1, 1), (-1, -1), 'LEFT'),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('ROWHEIGHT', (0, 0), (-1, -1), 1.2*inch),
    ]))

    elements.append(table)

    doc.build(elements)
    print(f"PDF generated successfully: {filename}")


if __name__ == "__main__":
    generate_pdf()

PDF generated successfully: Lilliput_QEC_Results_3_002.pdf
